<a href="https://colab.research.google.com/github/muneerh-fa/Sdaia-agentic-ai-systems-course/blob/main/assignments/assignment2/Copy_of_assignment2_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Install dependencies


In [ ]:
%pip install -q langchain langchain_openai langchain_community requests beautifulsoup4

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from langchain_openai import ChatOpenAI

**Set API key**

In [ ]:
try:
    from google.colab import userdata
    os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY").strip()
    print("OPENROUTER_API_KEY loaded successfully")
except Exception as e:
    print("Error loading API key:", e)

OPENROUTER_API_KEY loaded successfully


In [ ]:
model = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

In [ ]:
response = model.invoke("Answer in one short sentence: What is Ramadan?")
print(response.content)

Ramadan is the Islamic holy month of fasting, prayer, and reflection observed by Muslims worldwide.


## Tool 1 — Fetch URL Content

In [ ]:
import requests
from langchain.tools import tool

@tool
def fetch_url(url: str) -> str:
    """Fetch text content from a URL"""
    response = requests.get(url, timeout=10.0)
    response.raise_for_status()
    return response.text[:4000]


## Tool 2 — Internet Search

In [ ]:
from langchain.tools import tool
import requests
from bs4 import BeautifulSoup

@tool
def internet_search(query: str) -> str:
    """Search the internet and return top 3 results"""

    url = "https://duckduckgo.com/html/"
    params = {"q": query}

    response = requests.get(url, params=params)
    soup = BeautifulSoup(response.text, "html.parser")

    results = []

    for a in soup.select(".result__title a", limit=3):
        title = a.get_text(strip=True)
        link = a.get("href")
        results.append(f"{title} - {link}")

    return "\n".join(results)

## Agent System Prompt

In [ ]:
AGENT_PROMPT = """
You are an expert researcher.

Your job is to answer the user's question using information from the internet.

Steps you must follow:
1. Use internet_search to find relevant websites.
2. Select the top 3 search results.
3. Use fetch_url to retrieve the content of each website.
4. Combine the information and write a concise answer.

Always rely on multiple sources before answering.
Keep your final answer short and clear.
"""

## Create the Research Agent


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=model,
    tools=[internet_search, fetch_url],
    system_prompt=AGENT_PROMPT
)

## Test the Agent


In [ ]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is Agentic AI?"}]}
)

print(response)

{'messages': [HumanMessage(content='What is Agentic AI?', additional_kwargs={}, response_metadata={}, id='c5086d25-c7ed-4224-909f-8e3f6ad1b36f'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 175, 'prompt_tokens': 424, 'total_tokens': 599, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 167, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0, 'upstream_inference_prompt_cost': 0, 'upstream_inference_completions_cost': 0}}, 'model_provider': 'openai', 'model_name': 'nvidia/nemotron-3-nano-30b-a3b:free', 'system_fingerprint': None, 'id': 'gen-1773197351-jfJ8MWQHNlxSymoW88qd', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019cdacc-5a47-72e0-b286-51bdfa201f44-0', t